In [ ]:
import pandas as pd
%pip install -U nltk
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import re
import pandas as pd

path = r"all_ECB_speeches.csv"  # ou "all_ECB_speeches.csv"

data = pd.read_csv(
    path,
    sep="//",
    engine="python",
    encoding="utf-8-sig",
    on_bad_lines="warn"
)

data.insert(0, 'doc_id', range(1, len(data) + 1))
data.head()

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to C:\Users\Garance
[nltk_data]     Latieule/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\3109826370.py:12: ParserWarning: Skipping line 275: Expected 5 fields in line 275, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  data = pd.read_csv(
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\3109826370.py:12: ParserWarning: Skipping line 389: Expected 5 fields in line 389, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  data = pd.read_csv(
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\3109826370.py:12: ParserWarning: Skipping line 405: Expected 5 fields in line 405, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  data = pd.read_csv(
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\310

,doc_id,date,speakers,title,subtitle,contents
0,1,2025-09-30,Piero Cipollone,Innovating for stability: central bank money i...,"Welcome address by Piero Cipollone, Member of ...",SPEECH Innovating for stability: central ba...
1,2,2025-09-30,Christine Lagarde,Trade wars and central banks: lessons from 2025,"Keynote speech by Christine Lagarde, President...",SPEECH Trade wars and central banks: lesson...
2,3,2025-09-29,Piero Cipollone,"Digital euro: protecting our freedom, autonomy...","Keynote speech by Piero Cipollone, Member of t...",SPEECH Digital euro: protecting our freedom...
3,4,2025-09-26,Piero Cipollone,Preparing the future of payments and money: th...,"Keynote speech by Piero Cipollone, Member of t...",SPEECH Preparing the future of payments and...
4,5,2025-09-17,Piero Cipollone,What resilience takes: strengthening the finan...,"Keynote speech by Piero Cipollone, Member of t...",SPEECH What resilience takes: strengthening...


In [6]:
# Converting to string
text = data['contents']
text = text.astype("string")
text.head()

# --- NLTK punkt setup robuste + fallback ---
import os, ssl
import nltk

# 1) Dossier local pour éviter les soucis de droits
NLTK_DIR = os.path.join(os.getcwd(), ".nltk_data")
os.makedirs(NLTK_DIR, exist_ok=True)
if NLTK_DIR not in nltk.data.path:
    nltk.data.path.append(NLTK_DIR)

# 2) Téléchargements sûrs (gère SSL cassé)
try:
    _create_unverified_https_context = ssl._create_unverified_context
    ssl._create_default_https_context = _create_unverified_https_context  # si certifs manquants
except Exception:
    pass

def _ensure_resource(res_name):
    try:
        nltk.data.find(res_name)
        return True
    except LookupError:
        try:
            pkg = res_name.split("/")[1]
            nltk.download(pkg, download_dir=NLTK_DIR, quiet=True)
            nltk.data.find(res_name)  # vérifie
            return True
        except Exception:
            return False

has_punkt = _ensure_resource("tokenizers/punkt")
# certaines versions requièrent aussi punkt_tab
has_punkt_tab = _ensure_resource("tokenizers/punkt_tab")

# 3) Define a safe tokenizer (NLTK si dispo, sinon regex)
from typing import List
import re

if has_punkt:
    from nltk.tokenize import sent_tokenize as _nltk_sent_tokenize
    def safe_sent_tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
        return [s for s in _nltk_sent_tokenize(text) if s.strip()]
else:
    # Fallback simple (moins précis que NLTK, mais mieux que rien)
    _SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-ZÉÈÀÂÎÏÔÙÜÇ])')
    def safe_sent_tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
        parts = _SPLIT_RE.split(text)
        return [p.strip() for p in parts if p and p.strip()]


import pandas as pd
from nltk.tokenize import sent_tokenize

# Assure-toi que 'contents' est bien du texte
data['contents'] = data['contents'].fillna('').astype(str)

# Tokenize chaque discours en liste de phrases
tmp = data[['doc_id', 'speakers', 'date', 'contents']].copy()
tmp['sentences'] = tmp['contents'].apply(lambda t: sent_tokenize(t))

# Déplie une phrase par ligne
new_df_parsed = tmp.explode('sentences', ignore_index=True)

# Renomme et nettoie
new_df_parsed = new_df_parsed.rename(columns={
    'sentences': 'Parsed_Text',
    'speakers': 'Speaker',
    'date': 'Date',
    'doc_id': 'Doc_ID'
})
new_df_parsed = new_df_parsed[new_df_parsed['Parsed_Text'].str.strip().ne('')]

# Rang de phrase dans chaque discours (0,1,2,…) 
new_df_parsed['Sentence_Rank'] = new_df_parsed.groupby('Doc_ID').cumcount()

new_df_parsed.head()
new_df_parsed.tail()





,Doc_ID,Speaker,Date,contents,Parsed_Text,Sentence_Rank
327236,2681,Alexandre Lamfalussy,1997-02-07,Conference organised by the Hungarian Bankin...,My feeling is that you will be able to avoid s...,161
327237,2681,Alexandre Lamfalussy,1997-02-07,Conference organised by the Hungarian Bankin...,"But for a long time to come, most of your cust...",162
327238,2681,Alexandre Lamfalussy,1997-02-07,Conference organised by the Hungarian Bankin...,The comparative assessment of the costs and be...,163
327239,2681,Alexandre Lamfalussy,1997-02-07,Conference organised by the Hungarian Bankin...,"I hope that in ten years' time, if I am still ...",164
327240,2681,Alexandre Lamfalussy,1997-02-07,Conference organised by the Hungarian Bankin...,"Meanwhile, I wish you good luck - and the abil...",165


In [7]:
df = new_df_parsed

# 1) Neutraliser NaN et forcer en str
df['Parsed_Text'] = df['Parsed_Text'].fillna('').astype(str)

# 2) Enlever la ponctuation (garde lettres/chiffres/espaces)
df['Parsed_Text'] = df['Parsed_Text'].str.replace(r'[^\w\s]', '', regex=True)

# 3) Nettoyage espaces
df['Parsed_Text'] = df['Parsed_Text'].str.replace(r'\s+', ' ', regex=True).str.strip()

df = df[df['Parsed_Text'].str.len()>=20]


In [8]:

# Remove common redundant strings
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('cid173', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('digitized for fraser httpfraserstlouisfedorg federal reserve bank of st louis', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('UFB03', 'ffi', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0080U0099', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0080U0091', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0080U0094', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0080', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0099', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U0093', '', x))
df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('U009', '', x))

C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\699710337.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('cid173', '', x))
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\699710337.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Parsed_Text'] = df['Parsed_Text'].apply(lambda x: re.sub('digitized for fraser httpfraserstlouisfedorg federal reserve bank of st louis', '', x))
C:\Users\Garance Latieule\AppData\Loc

In [9]:
# We make all text lowercase
df['Parsed_Text'] = df['Parsed_Text'].map(str.lower)

C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_61352\636093244.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Parsed_Text'] = df['Parsed_Text'].map(str.lower)


In [10]:

# We remove all dublicate sentences
df = df.drop_duplicates(subset=['Parsed_Text'])
df = df.reset_index()
df = df.drop('index', axis=1)


df.head()

,Doc_ID,Speaker,Date,contents,Parsed_Text,Sentence_Rank
0,1,Piero Cipollone,2025-09-30,SPEECH Innovating for stability: central ba...,speech innovating for stability central bank m...,0
1,1,Piero Cipollone,2025-09-30,SPEECH Innovating for stability: central ba...,1 fast forward 12 months and the pace of trans...,1
2,1,Piero Cipollone,2025-09-30,SPEECH Innovating for stability: central ba...,innovative digital payment solutions and digit...,2
3,1,Piero Cipollone,2025-09-30,SPEECH Innovating for stability: central ba...,the opportunities and risks are being discusse...,3
4,1,Piero Cipollone,2025-09-30,SPEECH Innovating for stability: central ba...,2 so what does this mean for central banks,4


In [11]:
df = df.drop(columns=['contents'], errors='ignore')
df.head()

out_path = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_intermediate.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"[OK] Saved {len(df):,} rows -> {out_path}")


[OK] Saved 306,153 rows -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_intermediate.csv


pb car texte (speech) pas que en anglais

# --- Détection (langid) + traduction vers l'anglais pour%pip install -U deep-translator
%pip install -U deep_translator

import pandas as pd
import langid
from deep_translator import GoogleTranslator
from tqdm import tqdm
import os

# 1) Charge ton CSV (ou garde df si déjà en mémoire)
in_path  = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_intermediate.csv"
df = pd.read_csv(in_path)

TEXT_COL = "Parsed_Text"          # colonne texte d'entrée
OUT_LANG_COL = "lang"             # code langue détecté
OUT_TEXT_EN = "Parsed_Text_en"    # texte traduit / en anglais

if TEXT_COL not in df.columns:
    raise ValueError(f"La colonne '{TEXT_COL}' est introuvable. Colonnes: {list(df.columns)}")

# 2) Détection de langue avec langid (robuste aux NaN / vides)
def safe_detect(text):
    s = "" if pd.isna(text) else str(text).strip()
    if not s:
        return "und"
    try:
        # langid renvoie (code, score)
        code, _ = langid.classify(s)
        return code or "und"
    except Exception:
        return "und"

tqdm.pandas(desc="Détection de langue (langid)")
df[OUT_LANG_COL] = df[TEXT_COL].progress_apply(safe_detect)

# 3) Traduction conditionnelle -> EN (uniquement si non-EN et non-vide)
translator = GoogleTranslator(source="auto", target="en")
_cache = {}

def translate_if_needed(text, lang_code):
    s = "" if pd.isna(text) else str(text).strip()
    if not s or lang_code in {"en", "und"}:
        return text
    key = (s, lang_code)
    if key in _cache:
        return _cache[key]
    try:
        translated = translator.translate(s)
    except Exception:
        translated = text  # en cas d'erreur réseau/quota : conserve l'original
    _cache[key] = translated
    return translated

tqdm.pandas(desc="Traduction -> EN si nécessaire")
df[OUT_TEXT_EN] = [
    translate_if_needed(txt, lang)
    for txt, lang in tqdm(zip(df[TEXT_COL].tolist(), df[OUT_LANG_COL].tolist()), total=len(df))
]


# 5) Sauvegarde avec ordre de colonnes préféré + nouvelles colonnes en fin
out_dir = os.path.dirname(in_path)
out_path_en = os.path.join(out_dir, "pre_processed_ECB_intermediate_en.csv")

# Ordre préféré (selon ton pipeline). On inclut "contents" si présent seulement.
preferred_order = ["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text", "Sentence_Rank"]
ordered_cols = [c for c in preferred_order if c in df.columns]
# Le reste des colonnes (hors nos nouvelles)
tail_cols = [c for c in df.columns if c not in ordered_cols and c not in {OUT_LANG_COL, OUT_TEXT_EN}]
final_cols = ordered_cols + tail_cols + [OUT_LANG_COL, OUT_TEXT_EN]

#df.to_csv(out_path_en, index=False, encoding="utf-8-sig", columns=final_cols)
#print(f"\n[OK] Détecté (langid) & traduit si besoin : {len(df):,} lignes -> {out_path_en}")

In [ ]:
# %% [markdown]
# One-cell cleaner (stdlib only): stopwords + boilerplate + months/days + artifacts
# Protects economic vocabulary & negations; optional simple lemmatization; streams CSV.
#
# === Academic Justification ===
#
# ▸ Motivation:
#   Removing highly frequent or non-informative tokens is a standard step in text mining
#   to reduce dimensionality and emphasize discriminative terms.
#
# ▸ Information Retrieval (IR) foundation:
#   - Words with very high document frequency (DF) contribute little to discriminating
#     documents (Luhn, 1958; Salton & Buckley, 1988; Robertson, 2004).
#   - TF–IDF is built on this principle; removing these tokens explicitly reduces noise.
#
# ▸ Boilerplate & ceremonial language:
#   - Speech openings, greetings, job titles, and institutional phrases add structure but
#     little semantic value. Removing them improves retrieval precision and topic modeling
#     (Kohlschütter et al., 2010, *WWW*).
#
# ▸ Financial-domain adaptation:
#   - Generic stoplists are unreliable for finance because they remove domain-relevant words.
#     Therefore, we keep quantitative expressions and financial terms (Loughran & McDonald, 2011).
#
# ▸ Sentiment and FinBERT compatibility:
#   - Negations (“not”, “never”) must be preserved since they invert sentiment polarity
#     (Pang & Lee, 2008; Wiegand et al., 2010).
#   - Numeric data carry key economic information (growth rates, inflation, etc.) and should
#     not be removed (Chen et al., 2021, *FinBERT*).
#
# ▸ Linguistic normalization:
#   - Light lemmatization consolidates inflected forms without aggressive stemming,
#     improving token consistency while maintaining interpretability (Manning et al., 2008).
#
# ▸ In short:
#   This stoplist targets high-frequency grammatical and rhetorical words, as well as
#   ECB-specific boilerplate, while protecting negations, numbers, and economic vocabulary.
#   The goal is to enhance statistical and semantic signal quality for TF–IDF weighting
#   and FinBERT inference..

# %% 
import csv, re, sys
from typing import Iterable, Set, Optional

# ===================== 1) PARAMÈTRES À ADAPTER =====================
IN_CSV  = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_intermediate.csv"
OUT_CSV = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final.csv"
TEXT_COL = "Parsed_Text"      # colonne texte; None = auto-détection (Parsed_Text/contents/text/sentence)
NEW_COL  = "Parsed_Text_clean"
KEEP_NUMBERS = True           # garder les nombres (2, 2027, t2s…)
USE_LEMMA    = True           # lemmatisation simple maison (règles naïves)

# ===================== 2) LISTES DE MOTS =====================
# --- STOPLIST GÉANTE (fonctionnels, modaux, pronoms, prépositions,
#     connecteurs, verbes de rapport, verbes "légers", fillers, temps, etc.) ---
BASE_STOP = {
    # Articles / déterminants
    "the","a","an","some","any","each","every","either","neither","another","other","others",
    "such","this","that","these","those","same","own","former","latter",

    # Pronoms (pers., poss., démonstratifs, relatifs, indéfinis)
    "i","me","my","mine","myself","we","us","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself","she","her","hers","herself",
    "it","its","itself","they","them","their","theirs","themselves",
    "one","ones","someone","somebody","anyone","anybody","everyone","everybody",
    "noone","nobody","whose","which","who","whom","where","when","why","how",
    "whatever","whichever","whoever","whomever", "what",

    # Prépositions / particules
    "of","in","on","at","by","for","with","as","from","to","into","onto","over","under",
    "above","below","between","among","through","throughout","within","without","via",
    "across","along","around","behind","beyond","before","after","during","outside","inside",
    "about","regarding","concerning","per","versus","vs","amid","amidst","upon","off",

    # Conjonctions / connecteurs
    "and","or","but","so","than","then","that","if","though","although","because","since",
    "unless","until","while","whereas","whether","thus","therefore","hence","whereby",
    "thereby","therein","thereof","hereby","herein","furthermore","moreover","additionally",
    "meanwhile","nonetheless","nevertheless","instead","otherwise","likewise","similarly",

    # Adverbes fréquents / intensifieurs / fillers
    "very","too","also","just","still","only","even","ever","yet","again","almost","nearly",
    "roughly","around","approximately","about","rather","quite","pretty","fairly","truly",
    "indeed","simply","clearly","obviously","apparently","basically","essentially","largely",
    "generally","typically","commonly","mostly","mainly","primarily","overall","anyway","already",

    # Auxiliaires (BE / HAVE / DO) — (les négations seront normalisées en 'not')
    "be","am","is","are","was","were","been","being",
    "have","has","had","having",
    "do","does","did","doing",

    # Modaux
    "can","could","may","might","shall","should","will","would","must","ought",

    # Verbes de rapport / discours (peu informatifs)
    "say","says","said","saying",
    "tell","tells","told","telling",
    "state","states","stated","stating",
    "note","notes","noted","noting",
    "remark","remarks","remarked","remarking",
    "mention","mentions","mentioned","mentioning",
    "discuss","discusses","discussed","discussing",
    "address","addresses","addressed","addressing",
    "announce","announces","announced","announcing",
    "report","reports","reported","reporting",
    "explain","explains","explained","explaining",
    "highlight","highlights","highlighted","highlighting",
    "outline","outlines","outlined","outlining",
    "emphasize","emphasizes","emphasized","emphasizing",
    "stress","stresses","stressed","stressing",
    "point","points","pointed","pointing",
    "argue","argues","argued","arguing",
    "believe","believes","believed","believing",
    "think","thinks","thought","thinking",
    "consider","considers","considered","considering",
    "expect","expects","expected","expecting",
    "anticipate","anticipates","anticipated","anticipating",
    "estimate","estimates","estimated","estimating",
    "project","projects","projected","projecting",
    "assume","assumes","assumed","assuming",
    "suggest","suggests","suggested","suggesting",
    "propose","proposes","proposed","proposing",
    "recommend","recommends","recommended","recommending",
    "acknowledge","acknowledges","acknowledged","acknowledging",

    # Verbes "légers" / génériques
    "make","makes","made","making",
    "take","takes","took","taking",
    "give","gives","gave","giving",
    "get","gets","got","getting",
    "put","puts","put","putting",
    "keep","keeps","kept","keeping",
    "use","uses","used","using",
    "provide","provides","provided","providing",
    "include","includes","included","including",
    "present","presents","presented","presenting",
    "allow","allows","allowed","allowing",
    "enable","enables","enabled","enabling",
    "help","helps","helped","helping",
    "support","supports","supported","supporting",
    "maintain","maintains","maintained","maintaining",
    "improve","improves","improved","improving",
    "enhance","enhances","enhanced","enhancing",
    "deliver","delivers","delivered","delivering",
    "drive","drives","drove","driving",
    "lead","leads","led","leading",
    "bring","brings","brought","bringing",
    "show","shows","showed","showing",
    "look","looks","looked","looking",
    "work","works","worked","working",
    "move","moves","moved","moving",
    "go","goes","went","going",
    "come","comes","came","coming",
    "need","needs","needed","needing",
    "want","wants","wanted","wanting",
    "try","tries","tried","trying",
    "aim","aims","aimed","aiming",
    "plan","plans","planned","planning",
    "intend","intends","intended","intending",
    "continue","continues","continued","continuing",
    "remain","remains","remained","remaining",
    "appear","appears","appeared","appearing",
    "seem","seems","seemed","seeming",

    # Politesse / méta-discours
    "welcome","address","opening","closing","conclusion","thank","thanks","thanking",
    "good","morning","afternoon","evening","today","tonight","everyone","folks",
    "ladies","gentlemen","colleagues","friends","remarks","keynote",
    "conference","symposium","panel","session","event","meeting","host","hosted",
    "speech","speeches","statement","statements","introductory","press","release","blog","post",
    "member","members","chair","chairman","chairwoman","governor","president",
    "vice","deputy","board","executive","committee","department","division",

    # Mois / jours / temps génériques
    "january","february","march","april","may","june","july","august","september",
    "october","november","december",
    "monday","tuesday","wednesday","thursday","friday","saturday","sunday",
    "today","yesterday","tomorrow","tonight","morning","afternoon","evening",
    "week","weeks","month","months","year","years","quarter","quarters","semester","semesters",
    "day","days","daily","monthly","yearly","annually","quarterly",

    # Quantificateurs / mesure vagues
    "many","much","few","several","plenty","various","numerous","multiple",
    "more","most","less","least","greater","greatest","smaller","smallest",
    "somewhat","kind","sort","type","kinds","sorts","types",

    # Deixis / locatifs génériques
    "here","there","where","anywhere","everywhere","somewhere","nowhere","near","far",

    #divers
    "welcome","address","opening","closing","conclusion","thank","thanks","thanking",
    "good","morning","afternoon","evening","today","tonight","everyone",
    "ladies","gentlemen","colleagues","friends","remarks","keynote",
    "conference","symposium","panel","session","event","meeting","host","hosted",
    "speech","speeches","statement","statements","introductory","press","release","blog","post",
    "member","members","chair","chairman","chairwoman","governor","president",
    "vice","deputy","board","executive","committee","department","division",
    "let","me","us","want","like","say","ask","think","know","believe","going",
    "talk","speak","turn","point","points","make","made","making",
    "slide","slides","download","click","page","pages","copyright","video","pdf","transcript",
    "press","release","introductory","blog","post"
}



FINAL_STOP: Set[str] = BASE_STOP

# ===================== 3) NETTOYAGE =====================
TOKEN_RE = re.compile(r"[a-z0-9]+")

def simple_lemma(w: str) -> str:
    if len(w) <= 3: return w
    if w.endswith("ing") and len(w) > 5:
        base = w[:-3]
        if base.endswith(("iz","at")): return base + "e"
        if len(base) >= 2 and base[-1] == base[-2]: base = base[:-1]
        return base
    if w.endswith("ied") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("ed") and len(w) > 4:
        base = w[:-2]
        if base.endswith(("iz","at")): return base + "e"
        if len(base) >= 2 and base[-1] == base[-2]: base = base[:-1]
        return base
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("es") and len(w) > 3 and w[-3] in ("x","s","z","h"): return w[:-2]
    if w.endswith("s") and len(w) > 3 and not w.endswith("ss"): return w[:-1]
    return w

def clean_sentence(text: str, stopset: Set[str]=FINAL_STOP, keep_numbers: bool=True, use_lemma: bool=True) -> str:
    if not isinstance(text, str): text = "" if text is None else str(text)
    toks = TOKEN_RE.findall(text.lower())
    if not keep_numbers: toks = [t for t in toks if not t.isdigit()]
    toks = [t for t in toks if t not in stopset]
    if use_lemma: toks = [simple_lemma(t) for t in toks]
    toks = [t for t in toks if t]
    return " ".join(toks)

def process_csv(in_csv: str, out_csv: str, text_col: Optional[str]="Parsed_Text", new_col: str="Parsed_Text_clean",
                keep_numbers: bool=True, use_lemma: bool=True) -> int:
    """
    Lit in_csv, crée new_col, et ÉCRIT out_csv SANS la colonne text_col (Parsed_Text).
    """
    n = 0
    with open(in_csv, "r", newline="", encoding="utf-8") as fin:
        reader = csv.DictReader(fin)
        fields = list(reader.fieldnames or [])
        if not fields: print("[Erreur] CSV sans en-tête."); return 0
        if text_col is None or text_col not in fields:
            for cand in ("Parsed_Text","contents","text","sentence"):
                if cand in fields: text_col = cand; break
        if text_col not in fields: print(f"[Erreur] Colonne texte introuvable. Dispo: {fields}"); return 0

        # --- schéma de sortie: on retire la colonne brute et on garde la nettoyée
        fields_out = [f for f in fields if f != text_col]
        if new_col not in fields_out: fields_out.append(new_col)

        with open(out_csv, "w", newline="", encoding="utf-8") as fout:
            writer = csv.DictWriter(fout, fieldnames=fields_out)
            writer.writeheader()
            for row in reader:
                raw = row.get(text_col, "") or ""
                row[new_col] = clean_sentence(raw, stopset=FINAL_STOP,
                                              keep_numbers=keep_numbers, use_lemma=use_lemma)
                row.pop(text_col, None)            # <-- enlève Parsed_Text
                writer.writerow(row)
                n += 1
                if n % 100000 == 0: print(f"[Info] {n} lignes traitées...")
    return n

# ===================== 4) RUN =====================
n_lines = process_csv(
    IN_CSV, OUT_CSV, text_col=TEXT_COL, new_col=NEW_COL,
    keep_numbers=KEEP_NUMBERS, use_lemma=USE_LEMMA
)
print(f"[OK] {n_lines} lignes traitées. Écrit -> {OUT_CSV}")

# Aperçu (ne montre que la colonne nettoyée)
try:
    import pandas as pd
    preview = pd.read_csv(OUT_CSV, nrows=5)
    display(preview[[NEW_COL]] if NEW_COL in preview.columns else preview.head())
except Exception:
    pass

[Info] 100000 lignes traitées...
[Info] 200000 lignes traitées...
[Info] 300000 lignes traitées...
[OK] 306153 lignes traitées. Écrit -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final.csv


,Parsed_Text_clean
0,innovate stability central bank money digital ...
1,1 fast forward 12 pace transformation quicken
2,innovative digital payment solution digital as...
3,opportunity risk length
4,2 mean central bank


In [ ]:
# ===================== 5) DROP non-English + DROP Parsed_Text + SAVE =====================
import pandas as pd
from pathlib import Path

# --- paramètres (tu peux ajuster) ---
CONF_MIN = 0.90           # confiance minimale langue anglaise
MIN_SENT_PER_DOC = 3      # nombre min. de phrases par Doc_ID après filtrage
OUT_SUFFIX = "_en_noParsedText.csv"  # suffixe du nouveau fichier

# 1) Charger la sortie actuelle
df = pd.read_csv(r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final.csv")

# 2) Supprimer colonnes d'index fantômes
drop_unnamed = [c for c in df.columns if str(c).lower().startswith("unnamed")]
df = df.drop(columns=drop_unnamed, errors="ignore")

# 3) Détection de langue locale (langid)
try:
    import langid
    # Restreindre l'espace des langues pour fiabilité/vitesse
    langid.set_languages(['en','fr','de','es','it','pt','nl'])
except Exception as e:
    raise RuntimeError(
        "Le module 'langid' est requis. Installe-le d'abord:  pip install langid"
    ) from e

# Colonne source pour la détection (préfère le texte brut si présent)
SRC_COL = "Parsed_Text_clean"

# Classifier chaque phrase -> (lang, proba)
pred = df[SRC_COL].fillna("").astype(str).apply(lambda t: langid.classify(t or ""))
df["lang"] = [p[0] for p in pred]
df["lang_prob"] = [float(p[1]) for p in pred]

# 4) Filtrer: anglais avec confiance suffisante
df = df[(df["lang"] == "en") & (df["lang_prob"] >= CONF_MIN)].copy()

# 5) Option: exclure Doc_ID trop courts après filtrage
if "Doc_ID" in df.columns and "Sentence_Rank" in df.columns:
    counts = df.groupby("Doc_ID")["Sentence_Rank"].count()
    ok_docs = counts[counts >= MIN_SENT_PER_DOC].index
    df = df[df["Doc_ID"].isin(ok_docs)]


# 7) Tri + ordre de colonnes propre
sort_cols = [c for c in ["Doc_ID","Sentence_Rank"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)

desired = ["Doc_ID","Speaker","Date","contents","Sentence_Rank", NEW_COL]
ordered = [c for c in desired if c in df.columns]
rest = [c for c in df.columns if c not in ordered]
df = df[ordered + rest]

# 8) Sauvegarde finale (sans écraser l'original)
OUT_FINAL = Path(OUT_CSV).with_name(Path(OUT_CSV).stem + OUT_SUFFIX)
df.to_csv(OUT_FINAL, index=False, encoding="utf-8")
print(f"[OK] CSV anglais sans Parsed_Text -> {OUT_FINAL}  (lignes={len(df)}, colonnes={df.shape[1]})")



[OK] CSV anglais sans Parsed_Text -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final_en_noParsedText.csv  (lignes=3712, colonnes=7)
